In [1]:
from pathlib import Path
import sys
import os

PROJECT_ROOT = Path(r"C:\Users\User\Desktop\Vehicle_Damage_Detection")
os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

(PROJECT_ROOT / "src" / "__init__.py").touch(exist_ok=True)

print("Current folder:", Path.cwd())
print("src exists:", (PROJECT_ROOT / "src").exists())
print("model_inference exists:", (PROJECT_ROOT / "src" / "model_inference.py").exists())

Current folder: C:\Users\User\Desktop\Vehicle_Damage_Detection
src exists: True
model_inference exists: True


In [2]:
from src.model_inference import predict_damage_and_parts, get_best_damage, convert_parts_for_fusion
from src.fusion_logic import fuse_damage_and_part
from src.repair_rules import decide_repair_action
from src.cost_estimator import estimate_cost

print("Imports working")

Imports working


In [3]:
IMAGE_PATH = r"C:\Users\User\Desktop\Vehicle_Damage_Detection\test_images\sample_car.jpg"

print("Image exists:", Path(IMAGE_PATH).exists())

Image exists: True


In [ ]:
predictions = predict_damage_and_parts(IMAGE_PATH)

print("Damage detections:")
print(predictions["damage_detections"])

print("\nCar part detections:")
print(predictions["carpart_detections"])

best_damage = get_best_damage(predictions["damage_detections"])
parts_for_fusion = convert_parts_for_fusion(predictions["carpart_detections"])

if best_damage is None:
    print("\nNo damage detected.")
else:
    fusion_result = fuse_damage_and_part(best_damage, parts_for_fusion)

    severity = "moderate"  # temporary, later we connect archive2 severity model

    repair_result = decide_repair_action(
        damage_type=fusion_result["damage_type"],
        damaged_part=fusion_result["damaged_part"],
        severity=severity
    )

    cost_result = estimate_cost(
        brand="Toyota",
        model="Aqua",
        year="2018",
        damaged_part=fusion_result["damaged_part"],
        repair_action=repair_result["action"]
    )

    print("\nBest damage:")
    print(best_damage)

    print("\nFusion result:")
    print(fusion_result)

    print("\nRepair result:")
    print(repair_result)

    print("\nCost result:")
    print(cost_result)

Damage detections:
[{'class_name': 'dent', 'confidence': 0.8657, 'box': [126.15, 74.71, 296.92, 235.98]}]

Car part detections:
[{'class_name': 'wheel', 'confidence': 0.6523, 'box': [188.61, 201.23, 293.61, 387.59]}]

Best damage:
{'damage_type': 'dent', 'confidence': 0.8657, 'box': [126.15, 74.71, 296.92, 235.98]}

Fusion result:
{'damage_type': 'dent', 'damage_confidence': 0.8657, 'damaged_part': 'wheel', 'part_confidence': 0.6523, 'overlap_iou': 0.084, 'warning': None}

Repair result:
{'action': 'repair_without_replacement', 'replacement_option': 'not_required', 'reason': 'Dent may be repairable without replacing the part.'}

Cost result:
{'cost_available': True, 'cost_type': 'repair_cost', 'estimated_cost': 'LKR 5000 - 20000', 'note': 'This is an estimated range. Final price should be confirmed by a repair shop.'}


: 